## 1.Data Preprocessing

In [13]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

In [14]:
df = pd.read_csv("dataset.csv")
df.shape 

(2747, 3)

In [15]:
sample = df.sample(n=1200, random_state=42)

sample.to_csv("sample1200_before_washing.csv", index=False)

In [16]:
dw1 = pd.read_csv("sample1200_before_washing.csv")
dw1.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap'
}).set_table_styles(
    [dict(selector='th', props=[('text-align', 'left')])]
)
dw1.columns

Index(['id', 'context', 'teacher_response'], dtype='object')

In [17]:
annotation1 = dw1
annotation1.columns

Index(['id', 'context', 'teacher_response'], dtype='object')

In [18]:
annotation1 = annotation1[annotation1['teacher_response'].str.split().str.len() >= 5]
annotation1.sample(0)

,id,context,teacher_response


In [19]:
annotation1.shape

(746, 3)

In [20]:
annotation1 = annotation1[~annotation1['teacher_response'].str.contains(
    'choose|select|fill in|complete the sentence|look at|choose|fill in | complete | insert |next one| exercise | quick memory test',
    case=False,
    na=False
)]
annotation1 = annotation1[~annotation1['teacher_response'].str.contains('A\)|B\)|C\)|D\)', regex=True)]

In [21]:
annotation1 = annotation1[annotation1['context'].str.split().str.len() >= 20]
annotation1 = annotation1 = annotation1[annotation1['context'].str.split().str.len() < 80]
annotation1.shape

(679, 3)

In [22]:
annotation1.sample(0)

,id,context,teacher_response


In [23]:
mask_q = annotation1['context'].str.contains(r'\?', na=False)
mask_uncert =annotation1['context'].str.contains(
    r'not sure|maybe|i think|i guess|probably|seems|could it|is it|does it mean|am i right|right\?',
    case=False, regex=True, na=False
)
mask_teacher_support = annotation1['teacher_response'].str.contains(
    r'good question|well done|great|nice|no worries|don\'t worry|that\'s ok|exactly|you\'re right|almost|close|let\'s|we can|try|step by step|ok|why not|yes|!',
    case=False, regex=True, na=False
)
print(mask_q.sum(), mask_uncert.sum(), mask_teacher_support.sum())

561 292 396


In [24]:
mask_keep = mask_q & mask_uncert | mask_teacher_support
filtered = annotation1[mask_keep]

print(filtered.shape)

(503, 3)


In [25]:
import re

text = filtered['context'].fillna("").str.lower()

# 1) frustration（情绪挫败）
pat_fru = r"\b(" \
    r"frustrat\w*|annoy\w*|irritat\w*|upset\w*|" \
    r"angr\w*|furi\w*|exasperat\w*|agitat\w*|" \
    r"stress\w*|overwhelm\w*|exhaust\w*|tire\w*|" \
    r"fed-up|discourag\w*|dishearten\w*|demotivat\w*|" \
    r"hopeless\w*|helpless\w*|defeat\w*|desperat\w*|" \
    r"resentful\w*|impatient\w*|bitter\w*|tense\w*|" \
    r"unhappy\w*|dissatisf\w*|miserabl\w*|bother\w*" \
r")\b"

# 2) difficulty（任务难、卡住、做不出）
pat_diff = r"\b(" \
    r"difficult\w*|hard\w*|challeng\w*|complex\w*|" \
    r"complicat\w*|tough\w*|trick\w*|demand\w*|" \
    r"burdensom\w*|tax\w*|struggl\w*|fail\w*|" \
    r"stumbl\w*|falter\w*|block\w*|hinder\w*|" \
    r"obstruct\w*|hamper\w*|imped\w*|slow\w*|" \
    r"delay\w*|strain\w*|pressur\w*|overload\w*" \
r")\b"

# 3) confusion（不理解/不确定/求确认）
pat_conf = r"\b(" \
    r"confus\w*|uncertain\w*|unsur\w*|puzzl\w*|" \
    r"perplex\w*|bewilder\w*|mystif\w*|lost\w*|" \
    r"unclear\w*|ambigu\w*|misunderstand\w*|misinterpret\w*|" \
    r"misread\w*|misjudg\w*|misperceiv\w*|misconceiv\w*|" \
    r"misconstru\w*|misidentif\w*|misapprehend\w*" \
r")\b"

# 4) question（普通问句：有问号或 wh-word）
pat_q = r"(\?)|\b(why|how|what|when|where|which)\b"

mask_fru  = text.str.contains(pat_fru,  regex=True)
mask_diff = text.str.contains(pat_diff, regex=True)
mask_conf = text.str.contains(pat_conf, regex=True)
mask_q    = text.str.contains(pat_q,    regex=True)

# 优先级赋值
filtered_1 = filtered.copy()  # 你的500条
filtered_1["is_question"] = mask_q.astype(int)
filtered_1["is_confusion"] = mask_conf.astype(int)
filtered_1["is_difficulty"] = mask_diff.astype(int)
filtered_1["is_frustration"] = mask_fru.astype(int)

#加一个“组合标签”列
def make_labels(row):
    labels = []
    if row["is_question"]: labels.append("QUESTION")
    if row["is_confusion"]: labels.append("CONFUSION")
    if row["is_difficulty"]: labels.append("DIFFICULTY")
    if row["is_frustration"]: labels.append("FRUSTRATION")
    return "|".join(labels) if labels else "NEUTRAL"

filtered_1["context_labels"] = filtered_1.apply(make_labels, axis=1)

filtered_1["context_labels"].value_counts()




/var/folders/c8/vcr8t_yn0wq2rqnv1b7cqrvr0000gn/T/ipykernel_72262/3438213134.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_fru  = text.str.contains(pat_fru,  regex=True)
/var/folders/c8/vcr8t_yn0wq2rqnv1b7cqrvr0000gn/T/ipykernel_72262/3438213134.py:39: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_diff = text.str.contains(pat_diff, regex=True)
/var/folders/c8/vcr8t_yn0wq2rqnv1b7cqrvr0000gn/T/ipykernel_72262/3438213134.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_conf = text.str.contains(pat_conf, regex=True)
/var/folders/c8/vcr8t_yn0wq2rqnv1b7cqrvr0000gn/T/ipykernel_72262/3438213134.py:41: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the

context_labels
QUESTION                           383
QUESTION|DIFFICULTY                 42
NEUTRAL                             33
QUESTION|FRUSTRATION                18
QUESTION|CONFUSION                  11
DIFFICULTY                           6
QUESTION|DIFFICULTY|FRUSTRATION      5
QUESTION|CONFUSION|DIFFICULTY        3
FRUSTRATION                          2
Name: count, dtype: int64

In [26]:

filtered_1.sample(0)

,id,context,teacher_response,is_question,is_confusion,is_difficulty,is_frustration,context_labels


In [40]:
filtered_1 = filtered_1.reset_index(drop=True)
filtered_1.sample(0)

,id,context,teacher_response,is_question,is_confusion,is_difficulty,is_frustration,context_labels


In [28]:
filtered_1.shape

(503, 8)

In [29]:
filtered_1.to_csv("samples_after_1time_data_washing.csv")

## 2.Feature Engineering

In [30]:
df = pd.read_csv("samples_after_1time_data_washing.csv")
df.columns

Index(['Unnamed: 0', 'id', 'context', 'teacher_response', 'is_question',
       'is_confusion', 'is_difficulty', 'is_frustration', 'context_labels'],
      dtype='object')

In [41]:
dw2 = df[['id','context', 'teacher_response']].copy()
dw2.sample(0)

,id,context,teacher_response


In [42]:
dw2[['STG_Empathy', 'STG_Encouragement','STG_Explanation', 'STG_Directive']] = 0
dw2.sample(0)

,id,context,teacher_response,STG_Empathy,STG_Encouragement,STG_Explanation,STG_Directive


In [33]:
dw2.to_csv("teacher_strategy_annotation_v1.csv")

## 3.Pilot Annotation

In [34]:
df = pd.read_csv("samples_after_1time_data_washing.csv")
dw3 = df[['id', 'context', 'teacher_response']].copy()
dw3.shape

(503, 3)

In [43]:
pilot_dw3 = dw3.sample(20, random_state=42).reset_index(drop=True)
pilot_dw3[['support_score']] = 0 
pilot_dw3.sample(0)

,id,context,teacher_response,support_score


In [36]:

pilot_dw3.to_csv("pilot_annotation_20_samples.csv")

## 4.Final Annotation Dataset

In [37]:
df = pd.read_csv("samples_after_1time_data_washing.csv")
dw4 = df[['id', 'context', 'teacher_response']].copy()
dw4.shape

(503, 3)

In [44]:
dw4[['support_score']] = 0 
dw4.sample(0)

,id,context,teacher_response,support_score


In [39]:
dw4_1st_part = dw4[: 251].copy()
dw4_2nd_part = dw4[251 :].copy()
dw4.to_csv("annot_data_full.csv", index=True)
dw4_1st_part.to_csv("annot_data_part1.csv", index=True)
dw4_2nd_part.to_csv("annot_data_part2.csv", index=True)